In [1]:
import os
import sqlite3
from typing import Annotated, Literal
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    RemoveMessage,
)
from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState,
)
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from typing_extensions import TypedDict


In [6]:
# ============================================================
# Environment
# ============================================================

from pathlib import Path

env_file = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_file)

missing_keys = [
    key for key in ("GROQ_API_KEY", "TAVILY_API_KEY")
    if not os.getenv(key)
]
if missing_keys:
    raise ValueError(
        f"Missing {', '.join(missing_keys)}. Add them to {env_file} "
        "or set them in the notebook environment."
    )


# ============================================================
# Model
# ============================================================

model = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
)

In [7]:
# ============================================================
# Tools
# ============================================================

travel_search = TavilySearch(max_results=5)

tools = [travel_search]

model_with_tools = model.bind_tools(tools)
# ============================================================
# State
# ===========================================================
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [11]:
# ============================================================
# WITHOUT MEMORY
# ============================================================

from groq import APIConnectionError


def agent(state: State):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(State)

builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    tools_condition,
)

builder.add_edge("tools", "agent")

without_memory = builder.compile()


# ============================================================
# Run
# ============================================================

print("\n========== WITHOUT MEMORY ==========\n")

while True:

    user_message = input("You: ")

    if user_message.lower() in {"exit", "quit"}:
        break

    try:
        result = without_memory.invoke(
            {
                "messages": [
                    HumanMessage(content=user_message)
                ]
            }
        )
    except APIConnectionError as error:
        print(
            "Connection to Groq failed. Check DNS, firewall, VPN, or proxy settings "
            "and try again."
        )
        print(f"Details: {error}")
        continue

    print("Assistant:", result["messages"][-1].content)


========== WITHOUT MEMORY ==========

Assistant: 
Assistant: 


In [12]:
# ============================================================
# WITH MEMORY
# ============================================================

memory = MemorySaver()

with_memory = builder.compile(
    checkpointer=memory
)


config = {
    "configurable": {
        "thread_id": "conversation-1"
    }
}


print("\n========== WITH MEMORY ==========\n")

while True:

    user_message = input("You: ")

    if user_message.lower() in {"exit", "quit"}:
        break

    try:
        result = with_memory.invoke(
            {
                "messages": [
                    HumanMessage(content=user_message)
                ]
            },
            config=config,
        )
    except APIConnectionError as error:
        print(
            "Connection to Groq failed. Check DNS, firewall, VPN, or proxy settings "
            "and try again."
        )
        print(f"Details: {error}")
        continue

    print("Assistant:", result["messages"][-1].content)


========== WITH MEMORY ==========

Assistant: Nice to meet you, Amanda! How can I help you today?
Assistant: Your name is Amanda.
Assistant: Goodbye, Amanda! Have a great day.


In [13]:
class SummaryState(MessagesState):
    summary: str

In [14]:
def call_model(state: SummaryState):

    summary = state.get("summary", "")

    messages = state["messages"]

    if summary:
        messages = [
            SystemMessage(
                content=(
                    "Here is a summary of the earlier conversation. "
                    "Use it to maintain continuity:\n\n"
                    f"{summary}"
                )
            )
        ] + messages

    response = model_with_tools.invoke(messages)

    return {
        "messages": [response]
    }

In [15]:
def summarize_conversation(state: SummaryState):

    summary = state.get("summary", "")

    if summary:

        prompt = f"""
Update the existing conversation summary.

Existing summary:
{summary}

Preserve important facts, decisions, preferences, and context.
Remove irrelevant details.
"""

    else:

        prompt = """
Create a concise summary of this conversation.

Preserve:
- important facts
- user preferences
- decisions
- important context

Do not include unnecessary details.
"""

    messages = state["messages"] + [
        HumanMessage(content=prompt)
    ]

    response = model.invoke(messages)

    # Keep only the latest 2 messages
    messages_to_delete = [
        RemoveMessage(id=message.id)
        for message in state["messages"][:-2]
    ]

    return {
        "summary": response.content,
        "messages": messages_to_delete,
    }

In [16]:
def should_continue(
    state: SummaryState,
) -> Literal[
    "tools",
    "summarize",
    "__end__"
]:

    last_message = state["messages"][-1]

    # First priority: tool calling
    if getattr(last_message, "tool_calls", None):
        return "tools"

    # Summarize if conversation becomes large
    if len(state["messages"]) > 6:
        return "summarize"

    return END

In [17]:
summary_builder = StateGraph(SummaryState)

summary_builder.add_node(
    "conversation",
    call_model,
)

summary_builder.add_node(
    "tools",
    ToolNode(tools),
)

summary_builder.add_node(
    "summarize",
    summarize_conversation,
)

summary_builder.add_edge(
    START,
    "conversation",
)

summary_builder.add_conditional_edges(
    "conversation",
    should_continue,
)

summary_builder.add_edge(
    "tools",
    "conversation",
)

summary_builder.add_edge(
    "summarize",
    END,
)

In [18]:
summary_memory = MemorySaver()

graph_with_summary = summary_builder.compile(
    checkpointer=summary_memory
)

In [19]:
config = {
    "configurable": {
        "thread_id": "summary-chat-1"
    }
}


print("\n========== MEMORY + SUMMARY ==========\n")

while True:

    user_message = input("You: ")

    if user_message.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    result = graph_with_summary.invoke(
        {
            "messages": [
                HumanMessage(content=user_message)
            ]
        },
        config=config,
    )

    print(
        "Assistant:",
        result["messages"][-1].content
    )

    if result.get("summary"):
        print("\n[Conversation summary updated]\n")


========== MEMORY + SUMMARY ==========

Assistant: Nice to meet you, Amanda! How can I help you today?
Assistant: Hello Amanda! London has a fantastic range of hotels, from historic luxury palaces to modern budget-friendly stays.

Here are some top-rated options categorized by style and price point to help you get started:

### **Luxury & Iconic**
*   **The Dorchester:** A recently refurbished classic in Mayfair, known for its elegance and service.
*   **Claridge's:** One of London's most prestigious hotels, perfect if you want to be near high-end shopping on Bond Street.
*   **The Savoy:** A quintessential London hotel with a rich history and a prime location on the Strand.
*   **Shangri-La The Shard:** If you want the best views in the city, this hotel is located high up in The Shard skyscraper.
*   **Mandarin Oriental Hyde Park:** A great choice for families, located right next to Hyde Park and major museums.

### **Mid-Range & Boutique**
*   **St. Ermin's Hotel:** Highly rated and

In [28]:
# ============================================================
# PERSISTENT MEMORY
# ============================================================

class PersistentState(MessagesState):
    summary: str

In [29]:
conn = sqlite3.connect(
    "conversation.db",
    check_same_thread=False,
)

sqlite_memory = SqliteSaver(conn)

In [30]:
def persistent_call_model(
    state: PersistentState
):

    summary = state.get("summary", "")

    messages = state["messages"]

    if summary:

        messages = [
            SystemMessage(
                content=(
                    "Previous conversation summary:\n\n"
                    f"{summary}\n\n"
                    "Use this summary to maintain context."
                )
            )
        ] + messages

    response = model_with_tools.invoke(messages)

    return {
        "messages": [response]
    }

In [31]:
def persistent_summarize(
    state: PersistentState
):

    summary = state.get("summary", "")

    conversation = state["messages"]

    if summary:

        prompt = f"""
Update this conversation summary.

Previous summary:
{summary}

New conversation messages:
{conversation}

Keep important facts, decisions, preferences,
and context. Keep it concise.
"""

    else:

        prompt = f"""
Summarize this conversation concisely.

Keep important facts, decisions,
preferences, and context.

Conversation:
{conversation}
"""

    response = model.invoke(prompt)

    messages_to_delete = [
        RemoveMessage(id=message.id)
        for message in state["messages"][:-2]
    ]

    return {
        "summary": response.content,
        "messages": messages_to_delete,
    }

In [32]:
def persistent_router(
    state: PersistentState
) -> Literal[
    "tools",
    "summarize",
    "__end__"
]:

    last_message = state["messages"][-1]

    # Tool call
    if getattr(last_message, "tool_calls", None):
        return "tools"

    # Conversation too large
    if len(state["messages"]) > 3:
        return "summarize"

    return END

In [33]:
persistent_builder = StateGraph(
    PersistentState
)

persistent_builder.add_node(
    "conversation",
    persistent_call_model,
)

persistent_builder.add_node(
    "tools",
    ToolNode(tools),
)

persistent_builder.add_node(
    "summarize",
    persistent_summarize,
)

persistent_builder.add_edge(
    START,
    "conversation",
)

persistent_builder.add_conditional_edges(
    "conversation",
    persistent_router,
)

persistent_builder.add_edge(
    "tools",
    "conversation",
)

persistent_builder.add_edge(
    "summarize",
    END,
)

In [34]:
persistent_graph = persistent_builder.compile(
    checkpointer=sqlite_memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "persistent-chat-1"
    }
}


print("\n========== PERSISTENT MEMORY ==========\n")

while True:

    user_message = input("You: ")

    if user_message.lower() in {"exit", "quit"}:
        print("Chat ended.")
        break

    result = persistent_graph.invoke(
        {
            "messages": [
                HumanMessage(content=user_message)
            ]
        },
        config=config,
    )

    print(
        "Assistant:",
        result["messages"][-1].content
    )

    if result.get("summary"):
        print("\n[Summary updated]\n")